In [2]:
# ============================================================
# Step 1. 필요한 모듈 로딩
# ============================================================
from bs4 import BeautifulSoup
from selenium.webdriver.common.by import By
import time
import os
import re
import pandas as pd
import undetected_chromedriver as uc
from openpyxl import load_workbook
from openpyxl.styles import Alignment, Font, PatternFill
 
 
# ============================================================
# Step 2. 설정값
# ============================================================
URL_LIST = [
    'https://review.rakuten.co.jp/item/1/204591_10000422/1.1/?l2-id=item_review',
    'https://review.rakuten.co.jp/item/1/390266_10000002/1.1/?l2-id=item_review',
    'https://review.rakuten.co.jp/item/1/383198_10000451/1.1/?l2-id=item_review',
    'https://review.rakuten.co.jp/item/1/211527_10000085/1.1/?l2-id=item_review',
    'https://review.rakuten.co.jp/item/1/426554_10000007/1.1/?l2-id=item_review',
    'https://review.rakuten.co.jp/item/1/383198_10000231/1.1/?l2-id=item_review',
    'https://review.rakuten.co.jp/item/1/348380_10000046/1.1/?l2-id=item_review',
    'https://review.rakuten.co.jp/item/1/334402_10000099/1.1/?l2-id=item_review',
    'https://review.rakuten.co.jp/item/1/334402_10000093/1.1/?l2-id=item_review',
    'https://review.rakuten.co.jp/item/1/395258_10000116/1.1/?l2-id=item_review',
    'https://review.rakuten.co.jp/item/1/409735_10000007/1.1/?l2-id=item_review',
]
 
# 정렬별 뽑을 리뷰 수
TARGET_PER_SORT = 10   # ← 본수집 시 100으로 변경
 
# 저장 경로
f_dir = r"C:\py_temp\\"
 
# 정렬: (label, sort_param)
# sort=1: 新しい順(최신순) → 라쿠텐 기본
# sort=2: 評価の高い順(높은순)
# sort=3: 評価の低い順(낮은순)
SORT_MODES = [
    ('높은순', '2'),
    ('낮은순', '3'),
]
 
 
# ============================================================
# Step 3. 저장 폴더 설정
# ============================================================
n = time.localtime()
s = '%04d-%02d-%02d-%02d-%02d-%02d' % (
    n.tm_year, n.tm_mon, n.tm_mday, n.tm_hour, n.tm_min, n.tm_sec
)
platform = 'rakuten_review'
out_dir  = f_dir + s + '-' + platform + '\\'
os.makedirs(out_dir, exist_ok=True)
 
fx_name = out_dir + s + '-' + platform + '.xlsx'
fc_name = out_dir + s + '-' + platform + '.csv'
 
print("=" * 80)
print("라쿠텐 리뷰 크롤러")
print("=" * 80)
print(f"상품 수: {len(URL_LIST)}개  |  높은순 {TARGET_PER_SORT}개 + 낮은순 {TARGET_PER_SORT}개")
print(f"저장 경로: {out_dir}\n")
 
 
# ============================================================
# Step 4. 브라우저 실행
# ============================================================
driver = uc.Chrome(version_main=147)
driver.maximize_window()
time.sleep(2)
s_time = time.time()
 
 
# ============================================================
# Step 5. URL 생성 함수
# ============================================================
def make_page_url(base_url, page_no, sort_param):
    """
    base_url 예: https://review.rakuten.co.jp/item/1/204591_10000422/1.1/?l2-id=item_review
    → page2, sort=2: https://review.rakuten.co.jp/item/1/204591_10000422/2.1/?sort=2
    """
    match = re.search(r'/item/1/([^/]+)/', base_url)
    if not match:
        return base_url
    item_id = match.group(1)
    return f'https://review.rakuten.co.jp/item/1/{item_id}/{page_no}.1/?sort={sort_param}'
 
 
# ============================================================
# Step 6. 리뷰 파싱 함수 (실제 HTML 구조 기반)
# ============================================================
def parse_reviews(soup):
    """
    실제 라쿠텐 HTML 구조:
    - 리뷰 카드: <li> > div.spacer--1O71j.full-width--2Jy7N
    - 별점 숫자: span.number-wrapper-l--REPRg 안의 span 텍스트
    - 제목: div[class*="type-header"] > div.no-ellipsis 또는 직접 텍스트
    - 본문: div.no-ellipsis--2jV9- 또는 div.review-body--LpVR4
    """
    reviews = []
 
    # 리뷰 카드 = <li> 하위의 full-width div
    cards = soup.select('li > div.full-width--2Jy7N')
 
    for card in cards:
        # ---- 별점: number-wrapper 안 숫자 ----
        star = '없음'
        star_el = card.select_one('span.number-wrapper-l--REPRg span')
        if star_el:
            star = star_el.get_text(strip=True)
 
        # ---- 제목: type-header 클래스 포함 div ----
        title = ''
        # 제목은 헤더 스타일 div에 들어있음
        title_candidates = card.select('div[class*="type-header"]')
        for tc in title_candidates:
            t = tc.get_text(strip=True)
            if t:
                title = t
                break
 
        # ---- 본문: no-ellipsis--2jV9- 또는 review-body--LpVR4 ----
        body = ''
        body_el = card.select_one('div.no-ellipsis--2jV9-, div.review-body--LpVR4')
        if body_el:
            body = body_el.get_text(strip=True)
 
        # 제목 없으면 본문만, 둘 다 있으면 합치기
        if title and body:
            review_text = f'{title} {body}'
        elif title:
            review_text = title
        elif body:
            review_text = body
        else:
            continue  # 텍스트 없으면 스킵
 
        reviews.append({
            'star':   star,
            'title':  title,
            'body':   body,
            'review': review_text.strip(),
        })
 
    return reviews
 
 
# ============================================================
# Step 7. 상품명 추출 함수
# ============================================================
def get_product_name(soup):
    # 라쿠텐 리뷰 페이지 타이틀에서 상품명 추출
    try:
        # 브레드크럼 2번째 항목 (상품명)
        crumb = soup.select('li.c-breadcrumb-list-item a')
        if len(crumb) >= 2:
            return crumb[1].get_text(strip=True)[:80]
    except:
        pass
    try:
        title = soup.select_one('title')
        if title:
            t = title.get_text(strip=True)
            # 예: "상품명 | みんなのレビュー" 형태
            return t.split('|')[0].strip()[:80]
    except:
        pass
    return '상품명 없음'
 
 
# ============================================================
# Step 8. 상품별 리뷰 수집 함수
# ============================================================
def collect_reviews(driver, base_url, target, sort_label, sort_param):
    collected = []
    page_no   = 1
 
    while len(collected) < target:
        url = make_page_url(base_url, page_no, sort_param)
        print(f"    [{sort_label}] 페이지 {page_no} 접속 중...")
        driver.get(url)
        time.sleep(3)  # JS 렌더링 대기
 
        soup    = BeautifulSoup(driver.page_source, 'html.parser')
        reviews = parse_reviews(soup)
 
        if not reviews:
            print(f"    [{sort_label}] 페이지 {page_no}: 리뷰 없음 → 종료")
            break
 
        before = len(collected)
        for r in reviews:
            if len(collected) >= target:
                break
            collected.append(r)
 
        print(f"    [{sort_label}] 페이지 {page_no}: {len(reviews)}개 파싱 → 누적 {len(collected)}/{target}")
 
        if len(collected) >= target:
            break
 
        # 다음 페이지 확인: 리뷰가 30개 미만이면 마지막 페이지
        if len(reviews) < 30:
            print(f"    [{sort_label}] 마지막 페이지 (리뷰 {len(reviews)}개 < 30)")
            break
 
        page_no += 1
        time.sleep(1.5)
 
    return collected
 
 
# ============================================================
# Step 9. 전체 수집 루프
# ============================================================
all_rows = []
 
for prod_idx, base_url in enumerate(URL_LIST, start=1):
    print(f"\n{'='*65}")
    print(f"[{prod_idx}/{len(URL_LIST)}] 상품 처리 중")
    print(f"  URL: {base_url}")
 
    # 상품명 가져오기
    driver.get(base_url)
    time.sleep(3)
    soup = BeautifulSoup(driver.page_source, 'html.parser')
    product_name = get_product_name(soup)
    print(f"  상품명: {product_name}")
 
    for sort_label, sort_param in SORT_MODES:
        print(f"\n  ▶ 정렬: {sort_label} ({TARGET_PER_SORT}개 목표)")
        reviews = collect_reviews(driver, base_url, TARGET_PER_SORT, sort_label, sort_param)
 
        for rank, r in enumerate(reviews, start=1):
            all_rows.append({
                '상품번호': prod_idx,
                '상품명':   product_name,
                '상품URL':  base_url,
                '정렬':     f'{sort_label} {rank}번째',
                '번호':     rank,
                '별점':     r['star'],
                '리뷰내용': r['review'],
            })
 
        print(f"  ✅ {sort_label}: {len(reviews)}개 수집 완료")
 
print(f"\n{'='*65}")
print(f"전체 수집 완료: {len(all_rows)}건\n")
 
 
# ============================================================
# Step 10. 데이터프레임 생성 및 저장
# ============================================================
df = pd.DataFrame(all_rows, columns=[
    '상품번호', '상품명', '상품URL', '정렬', '번호', '별점', '리뷰내용'
])
df.index = df.index + 1
 
df.to_csv(fc_name, index=True, encoding='utf-8-sig')
df.to_excel(fx_name, index=True, engine='openpyxl')
print(f"CSV  저장: {fc_name}")
print(f"XLSX 저장: {fx_name}")
 
 
# ============================================================
# Step 11. xlsx 서식 정리
# ============================================================
wb = load_workbook(fx_name)
ws = wb['Sheet1']
 
col_widths = {
    'A': 5,   # index
    'B': 8,   # 상품번호
    'C': 35,  # 상품명
    'D': 55,  # 상품URL
    'E': 20,  # 정렬
    'F': 8,   # 번호
    'G': 8,   # 별점
    'H': 90,  # 리뷰내용
}
for col, width in col_widths.items():
    ws.column_dimensions[col].width = width
 
# 헤더 스타일
header_fill = PatternFill(fill_type='solid', fgColor='4472C4')
header_font = Font(bold=True, color='FFFFFF')
for cell in ws[1]:
    cell.fill = header_fill
    cell.font = header_font
    cell.alignment = Alignment(horizontal='center', vertical='center')
 
# 내용 셀
for row in ws.iter_rows(min_row=2):
    for cell in row:
        cell.alignment = Alignment(wrap_text=True, vertical='top')
 
# 높은순/낮은순 행 배경색 구분
for row in ws.iter_rows(min_row=2, max_col=8):
    sort_cell = row[3].value  # '정렬' 컬럼
    if sort_cell and '낮은순' in str(sort_cell):
        for cell in row:
            cell.fill = PatternFill(fill_type='solid', fgColor='FFF2CC')  # 연노란색
 
ws.freeze_panes = 'A2'  # 헤더 고정
wb.save(fx_name)
print("서식 정리 완료!")
 
 
# ============================================================
# Step 12. 요약
# ============================================================
e_time = time.time()
t_time = e_time - s_time
 
print("\n" + "=" * 80)
print(f"(1) 상품 수: {len(URL_LIST)}개")
print(f"(2) 총 수집 리뷰: {len(all_rows)}건")
print(f"(3) 총 소요시간: {round(t_time, 1)}초")
print(f"(4) CSV  저장: {fc_name}")
print(f"(5) XLSX 저장: {fx_name}")
print("=" * 80)
 

라쿠텐 리뷰 크롤러
상품 수: 11개  |  높은순 10개 + 낮은순 10개
저장 경로: C:\py_temp\\2026-04-28-14-45-23-rakuten_review\


[1/11] 상품 처리 중
  URL: https://review.rakuten.co.jp/item/1/204591_10000422/1.1/?l2-id=item_review
  상품명: 【楽天市場】ニキビ 吹き出物 肌荒れ 【楽天ランキング1位】 ティートリークリーム ティーツリークリーム オーガニック 保湿クリーム 大人ニキビ 思春期ニキビ 

  ▶ 정렬: 높은순 (10개 목표)
    [높은순] 페이지 1 접속 중...
    [높은순] 페이지 1: 30개 파싱 → 누적 10/10
  ✅ 높은순: 10개 수집 완료

  ▶ 정렬: 낮은순 (10개 목표)
    [낮은순] 페이지 1 접속 중...
    [낮은순] 페이지 1: 30개 파싱 → 누적 10/10
  ✅ 낮은순: 10개 수집 완료

[2/11] 상품 처리 중
  URL: https://review.rakuten.co.jp/item/1/390266_10000002/1.1/?l2-id=item_review
  상품명: 【楽天市場】idio イディオ 北海道PDRNクリーム 母の日 ギフト CICA セラミド 高保湿 乾燥 敏感肌 保湿クリーム ベタつかない 55g 日本製(F

  ▶ 정렬: 높은순 (10개 목표)
    [높은순] 페이지 1 접속 중...
    [높은순] 페이지 1: 30개 파싱 → 누적 10/10
  ✅ 높은순: 10개 수집 완료

  ▶ 정렬: 낮은순 (10개 목표)
    [낮은순] 페이지 1 접속 중...
    [낮은순] 페이지 1: 30개 파싱 → 누적 10/10
  ✅ 낮은순: 10개 수집 완료

[3/11] 상품 처리 중
  URL: https://review.rakuten.co.jp/item/1/383198_10000451/1.1/?l2-id=item_review
  상품명: 【楽天市場】ドレススノー ナイトクリーム 医薬部

In [ ]:
# ============================================================
# Step 1. 필요한 모듈 로딩
# ============================================================
from bs4 import BeautifulSoup
from selenium.webdriver.common.by import By
import time
import os
import re
import pandas as pd
import undetected_chromedriver as uc
from openpyxl import load_workbook, Workbook
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter
 
 
# ============================================================
# Step 2. 설정값
# ============================================================
URL_LIST = [
    'https://review.rakuten.co.jp/item/1/204591_10000422/1.1/?l2-id=item_review',
    'https://review.rakuten.co.jp/item/1/390266_10000002/1.1/?l2-id=item_review',
    'https://review.rakuten.co.jp/item/1/383198_10000451/1.1/?l2-id=item_review',
    'https://review.rakuten.co.jp/item/1/211527_10000085/1.1/?l2-id=item_review',
    'https://review.rakuten.co.jp/item/1/426554_10000007/1.1/?l2-id=item_review',
    'https://review.rakuten.co.jp/item/1/383198_10000231/1.1/?l2-id=item_review',
    'https://review.rakuten.co.jp/item/1/348380_10000046/1.1/?l2-id=item_review',
    'https://review.rakuten.co.jp/item/1/334402_10000099/1.1/?l2-id=item_review',
    'https://review.rakuten.co.jp/item/1/334402_10000093/1.1/?l2-id=item_review',
    'https://review.rakuten.co.jp/item/1/395258_10000116/1.1/?l2-id=item_review',
    'https://review.rakuten.co.jp/item/1/409735_10000007/1.1/?l2-id=item_review',
]
 
# 정렬별 뽑을 리뷰 수 (높은순 100 + 낮은순 100 = 총 200)
TARGET_PER_SORT = 10   # ← 본수집 시 100으로 변경
 
# 저장 경로
f_dir = r"C:\py_temp\\"
 
# 정렬: (label, sort_param)
# sort=2: 評価の高い順(높은순), sort=3: 評価の低い順(낮은순)
SORT_MODES = [
    ('높은순', '2'),
    ('낮은순', '3'),
]
 
 
# ============================================================
# Step 3. 저장 폴더 설정
# ============================================================
n = time.localtime()
s = '%04d-%02d-%02d-%02d-%02d-%02d' % (
    n.tm_year, n.tm_mon, n.tm_mday, n.tm_hour, n.tm_min, n.tm_sec
)
platform = 'rakuten_review'
out_dir  = f_dir + s + '-' + platform + '\\'
os.makedirs(out_dir, exist_ok=True)
fx_name = out_dir + s + '-' + platform + '.xlsx'
 
print("=" * 80)
print("라쿠텐 리뷰 크롤러")
print("=" * 80)
print(f"상품 수: {len(URL_LIST)}개  |  높은순 {TARGET_PER_SORT}개 + 낮은순 {TARGET_PER_SORT}개")
print(f"저장 경로: {out_dir}\n")
 
 
# ============================================================
# Step 4. 브라우저 실행
# ============================================================
driver = uc.Chrome(version_main=147)
driver.maximize_window()
time.sleep(2)
s_time = time.time()
 
 
# ============================================================
# Step 5. URL 생성 함수
# ============================================================
def make_page_url(base_url, page_no, sort_param):
    match = re.search(r'/item/1/([^/]+)/', base_url)
    if not match:
        return base_url
    item_id = match.group(1)
    return f'https://review.rakuten.co.jp/item/1/{item_id}/{page_no}.1/?sort={sort_param}'
 
 
# ============================================================
# Step 6. 리뷰 파싱 함수
# ============================================================
def parse_reviews(soup):
    reviews = []
    cards = soup.select('li > div.full-width--2Jy7N')
 
    for card in cards:
        # 별점
        star = '없음'
        star_el = card.select_one('span.number-wrapper-l--REPRg span')
        if star_el:
            star = star_el.get_text(strip=True)
 
        # 제목
        title = ''
        for tc in card.select('div[class*="type-header"]'):
            t = tc.get_text(strip=True)
            if t:
                title = t
                break
 
        # 본문
        body = ''
        body_el = card.select_one('div.no-ellipsis--2jV9-, div.review-body--LpVR4')
        if body_el:
            body = body_el.get_text(strip=True)
 
        # 제목 + 본문 합치기
        if title and body:
            review_text = f'{title} {body}'
        elif title:
            review_text = title
        elif body:
            review_text = body
        else:
            continue
 
        reviews.append({
            'star':   star,
            'review': review_text.strip(),
        })
 
    return reviews
 
 
# ============================================================
# Step 7. 상품명 추출 함수
# ============================================================
def get_product_name(soup):
    try:
        crumb = soup.select('li.c-breadcrumb-list-item a')
        if len(crumb) >= 2:
            return crumb[1].get_text(strip=True)[:80]
    except:
        pass
    try:
        title = soup.select_one('title')
        if title:
            return title.get_text(strip=True).split('|')[0].strip()[:80]
    except:
        pass
    return '상품명 없음'
 
 
# ============================================================
# Step 8. 상품별 리뷰 수집 함수
# ============================================================
def collect_reviews(driver, base_url, target, sort_label, sort_param):
    collected = []
    page_no   = 1
 
    while len(collected) < target:
        url = make_page_url(base_url, page_no, sort_param)
        print(f"    [{sort_label}] 페이지 {page_no} 접속 중...")
        driver.get(url)
        time.sleep(3)
 
        soup    = BeautifulSoup(driver.page_source, 'html.parser')
        reviews = parse_reviews(soup)
 
        if not reviews:
            print(f"    [{sort_label}] 페이지 {page_no}: 리뷰 없음 → 종료")
            break
 
        for r in reviews:
            if len(collected) >= target:
                break
            collected.append(r)
 
        print(f"    [{sort_label}] 페이지 {page_no}: {len(reviews)}개 파싱 → 누적 {len(collected)}/{target}")
 
        if len(collected) >= target:
            break
        if len(reviews) < 30:
            print(f"    [{sort_label}] 마지막 페이지")
            break
 
        page_no += 1
        time.sleep(1.5)
 
    return collected
 
 
# ============================================================
# Step 9. 전체 수집 루프 + 시트별 저장
# ============================================================
wb = Workbook()
wb.remove(wb.active)  # 기본 시트 제거
 
# 스타일 상수
HEADER_FILL    = PatternFill(fill_type='solid', fgColor='4472C4')
HEADER_FONT    = Font(bold=True, color='FFFFFF', size=11)
TITLE_FILL     = PatternFill(fill_type='solid', fgColor='D9E1F2')
TITLE_FONT     = Font(bold=True, size=12)
LOW_FILL       = PatternFill(fill_type='solid', fgColor='FFF2CC')  # 낮은순 행 배경
CENTER_ALIGN   = Alignment(horizontal='center', vertical='center', wrap_text=True)
LEFT_ALIGN_TOP = Alignment(horizontal='left',   vertical='top',    wrap_text=True)
 
COLS = ['리뷰아이디', '별점', '리뷰내용', '정렬방식']
COL_WIDTHS = [12, 8, 90, 12]
 
for prod_idx, base_url in enumerate(URL_LIST, start=1):
    print(f"\n{'='*65}")
    print(f"[{prod_idx}/{len(URL_LIST)}] 상품 처리 중")
    print(f"  URL: {base_url}")
 
    # 상품명
    driver.get(base_url)
    time.sleep(3)
    soup = BeautifulSoup(driver.page_source, 'html.parser')
    product_name = get_product_name(soup)
    print(f"  상품명: {product_name}")
 
    # 시트 이름: 번호_상품명 (엑셀 시트명 31자 제한 + 특수문자 불가)
    safe_name = re.sub(r'[\\/*?:\[\]]', '', product_name)[:25]
    sheet_name = f'{prod_idx}_{safe_name}'
 
    ws = wb.create_sheet(title=sheet_name)
 
    # ── Row 1: 상품명 (A1 전체 병합) ──────────────────────────
    ws.merge_cells(f'A1:{get_column_letter(len(COLS))}1')
    title_cell = ws['A1']
    title_cell.value     = product_name
    title_cell.fill      = TITLE_FILL
    title_cell.font      = TITLE_FONT
    title_cell.alignment = CENTER_ALIGN
    ws.row_dimensions[1].height = 28
 
    # ── Row 2: 컬럼 헤더 ──────────────────────────────────────
    for col_i, col_name in enumerate(COLS, start=1):
        cell = ws.cell(row=2, column=col_i, value=col_name)
        cell.fill      = HEADER_FILL
        cell.font      = HEADER_FONT
        cell.alignment = CENTER_ALIGN
    ws.row_dimensions[2].height = 22
 
    # 컬럼 너비 설정
    for col_i, width in enumerate(COL_WIDTHS, start=1):
        ws.column_dimensions[get_column_letter(col_i)].width = width
 
    # ── 데이터 수집 & 기록 ────────────────────────────────────
    current_row = 3
    review_id   = 1  # 리뷰아이디: 높은순 1~N, 낮은순 N+1~2N (연속)
 
    for sort_label, sort_param in SORT_MODES:
        print(f"\n  ▶ 정렬: {sort_label} ({TARGET_PER_SORT}개 목표)")
        reviews = collect_reviews(driver, base_url, TARGET_PER_SORT, sort_label, sort_param)
 
        for r in reviews:
            row_fill = LOW_FILL if sort_label == '낮은순' else None
 
            values = [review_id, r['star'], r['review'], sort_label]
            for col_i, val in enumerate(values, start=1):
                cell = ws.cell(row=current_row, column=col_i, value=val)
                cell.alignment = CENTER_ALIGN if col_i != 3 else LEFT_ALIGN_TOP
                if row_fill:
                    cell.fill = row_fill
 
            ws.row_dimensions[current_row].height = 60
            current_row += 1
            review_id   += 1
 
        print(f"  ✅ {sort_label}: {len(reviews)}개 → 리뷰아이디 {review_id - len(reviews)}~{review_id - 1}")
 
    print(f"  시트 '{sheet_name}' 완료 (총 {review_id - 1}개)")
 
# 헤더 행 고정 (행 1~2 고정)
for ws in wb.worksheets:
    ws.freeze_panes = 'A3'
 
wb.save(fx_name)
 
# ============================================================
# Step 10. 요약
# ============================================================
e_time = time.time()
t_time = e_time - s_time
 
print("\n" + "=" * 80)
print(f"(1) 상품 수: {len(URL_LIST)}개  → 시트 {len(wb.worksheets)}개 생성")
print(f"(2) 총 소요시간: {round(t_time, 1)}초")
print(f"(3) XLSX 저장: {fx_name}")
print("=" * 80)
 

라쿠텐 리뷰 크롤러
상품 수: 11개  |  높은순 10개 + 낮은순 10개
저장 경로: C:\py_temp\\2026-04-28-14-57-24-rakuten_review\


[1/11] 상품 처리 중
  URL: https://review.rakuten.co.jp/item/1/204591_10000422/1.1/?l2-id=item_review
  상품명: 【楽天市場】ニキビ 吹き出物 肌荒れ 【楽天ランキング1位】 ティートリークリーム ティーツリークリーム オーガニック 保湿クリーム 大人ニキビ 思春期ニキビ 

  ▶ 정렬: 높은순 (10개 목표)
    [높은순] 페이지 1 접속 중...
    [높은순] 페이지 1: 30개 파싱 → 누적 10/10
  ✅ 높은순: 10개 → 리뷰아이디 1~10

  ▶ 정렬: 낮은순 (10개 목표)
    [낮은순] 페이지 1 접속 중...
    [낮은순] 페이지 1: 30개 파싱 → 누적 10/10
  ✅ 낮은순: 10개 → 리뷰아이디 11~20
  시트 '1_【楽天市場】ニキビ 吹き出物 肌荒れ 【楽天ランキ' 완료 (총 20개)

[2/11] 상품 처리 중
  URL: https://review.rakuten.co.jp/item/1/390266_10000002/1.1/?l2-id=item_review
  상품명: 【楽天市場】idio イディオ 北海道PDRNクリーム 母の日 ギフト CICA セラミド 高保湿 乾燥 敏感肌 保湿クリーム ベタつかない 55g 日本製(F

  ▶ 정렬: 높은순 (10개 목표)
    [높은순] 페이지 1 접속 중...
    [높은순] 페이지 1: 30개 파싱 → 누적 10/10
  ✅ 높은순: 10개 → 리뷰아이디 1~10

  ▶ 정렬: 낮은순 (10개 목표)
    [낮은순] 페이지 1 접속 중...
    [낮은순] 페이지 1: 30개 파싱 → 누적 10/10
  ✅ 낮은순: 10개 → 리뷰아이디 11~20
  시트 '2_【楽天市場】idio イディオ 北海道PDRNクリ' 완료 (총 20개)

[3/11]

In [5]:
# ============================================================
# Step 1. 필요한 모듈 로딩
# ============================================================
from bs4 import BeautifulSoup
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import os
import re
import undetected_chromedriver as uc
from openpyxl import Workbook
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter


# ============================================================
# Step 2. 설정값
# ============================================================
URL_LIST = [
    'https://review.rakuten.co.jp/item/1/204591_10000422/1.1/?l2-id=item_review',
    'https://review.rakuten.co.jp/item/1/390266_10000002/1.1/?l2-id=item_review',
    'https://review.rakuten.co.jp/item/1/383198_10000451/1.1/?l2-id=item_review',
    'https://review.rakuten.co.jp/item/1/211527_10000085/1.1/?l2-id=item_review',
    'https://review.rakuten.co.jp/item/1/426554_10000007/1.1/?l2-id=item_review',
    'https://review.rakuten.co.jp/item/1/383198_10000231/1.1/?l2-id=item_review',
    'https://review.rakuten.co.jp/item/1/348380_10000046/1.1/?l2-id=item_review',
    'https://review.rakuten.co.jp/item/1/334402_10000099/1.1/?l2-id=item_review',
    'https://review.rakuten.co.jp/item/1/334402_10000093/1.1/?l2-id=item_review',
    'https://review.rakuten.co.jp/item/1/395258_10000116/1.1/?l2-id=item_review',
    'https://review.rakuten.co.jp/item/1/409735_10000007/1.1/?l2-id=item_review',
]

TARGET_PER_SORT = 10   # ← 본수집 시 100으로 변경
f_dir = r"C:\py_temp\\"

# select value: '1' = レビュー評価順(고평가순)
SORT_VALUE = '1'


# ============================================================
# Step 3. 저장 폴더 & 파일명 설정
# ============================================================
n = time.localtime()
s = '%04d-%02d-%02d-%02d-%02d-%02d' % (
    n.tm_year, n.tm_mon, n.tm_mday, n.tm_hour, n.tm_min, n.tm_sec
)
platform = 'rakuten_review'
out_dir  = f_dir + s + '-' + platform + '\\'
os.makedirs(out_dir, exist_ok=True)
fx_name = out_dir + s + '-' + platform + '.xlsx'

print("=" * 80)
print("라쿠텐 리뷰 크롤러 v5")
print("=" * 80)
print(f"상품 수: {len(URL_LIST)}개  |  고평가순 {TARGET_PER_SORT}개 + 저평가순 {TARGET_PER_SORT}개")
print(f"저장 경로: {out_dir}\n")


# ============================================================
# Step 4. 브라우저 실행
# ============================================================
driver = uc.Chrome(version_main=147)
driver.maximize_window()
time.sleep(2)
s_time = time.time()
wait = WebDriverWait(driver, 15)


# ============================================================
# Step 5. 핵심: 정렬 select 클릭 함수
# ============================================================
def apply_sort(driver, sort_value):
    """
    1) select 요소가 로드될 때까지 대기
    2) select를 클릭해서 드롭다운 열기
    3) 원하는 option 클릭
    4) 페이지 리로드 대기
    """
    try:
        # select 요소 대기
        select_el = wait.until(
            EC.presence_of_element_located(
                (By.CSS_SELECTOR, 'select.link-dropdown--3uu_U')
            )
        )
        # 화면에 보이도록 스크롤
        driver.execute_script("arguments[0].scrollIntoView({block:'center'});", select_el)
        time.sleep(0.5)

        # ① select 클릭 (드롭다운 열기)
        driver.execute_script("arguments[0].click();", select_el)
        time.sleep(0.5)

        # ② 원하는 option 클릭
        option = select_el.find_element(
            By.CSS_SELECTOR, f'option[value="{sort_value}"]'
        )
        driver.execute_script("arguments[0].click();", option)
        time.sleep(0.3)

        # ③ change 이벤트 강제 발생 (React 기반 UI 대응)
        driver.execute_script(
            f"""
            var sel = arguments[0];
            sel.value = '{sort_value}';
            sel.dispatchEvent(new Event('change', {{ bubbles: true }}));
            """,
            select_el
        )

        # ④ 페이지 리렌더링 대기
        time.sleep(3)
        print(f"    ✅ 정렬 적용 완료 (value={sort_value})")
        return True

    except Exception as e:
        print(f"    ⚠ 정렬 select 실패: {e}")
        return False


# ============================================================
# Step 6. URL 생성 함수 (정렬 적용된 페이지)
# ============================================================
def make_page_url(base_url, page_no):
    match = re.search(r'/item/1/([^/]+)/', base_url)
    if not match:
        return base_url
    item_id = match.group(1)
    # 페이지 이동은 URL로, 정렬은 페이지 내 select로
    return f'https://review.rakuten.co.jp/item/1/{item_id}/{page_no}.1/'


# ============================================================
# Step 7. 리뷰 파싱 함수
# ============================================================
def parse_reviews(soup):
    reviews = []
    cards = soup.select('li > div.full-width--2Jy7N')

    for card in cards:
        # 별점
        star = '없음'
        star_el = card.select_one('span.number-wrapper-l--REPRg span')
        if star_el:
            star = star_el.get_text(strip=True)

        # 제목
        title = ''
        for tc in card.select('div[class*="type-header"]'):
            t = tc.get_text(strip=True)
            if t:
                title = t
                break

        # 본문
        body = ''
        body_el = card.select_one('div.no-ellipsis--2jV9-, div.review-body--LpVR4')
        if body_el:
            body = body_el.get_text(strip=True)

        if title and body:
            review_text = f'{title} {body}'
        elif title:
            review_text = title
        elif body:
            review_text = body
        else:
            continue

        reviews.append({'star': star, 'review': review_text.strip()})

    return reviews


# ============================================================
# Step 8. 상품명 추출
# ============================================================
def get_product_name(soup):
    try:
        crumb = soup.select('li.c-breadcrumb-list-item a')
        if len(crumb) >= 2:
            return crumb[1].get_text(strip=True)[:80]
    except:
        pass
    try:
        title = soup.select_one('title')
        if title:
            return title.get_text(strip=True).split('|')[0].strip()[:80]
    except:
        pass
    return '상품명 없음'


# ============================================================
# Step 9. 마지막 페이지 번호 파악
# ============================================================
def get_last_page(soup):
    last_page = 1
    # href 안의 페이지 번호 패턴으로 최댓값 추출
    for a in soup.select('a[href]'):
        href = a.get('href', '')
        m = re.search(r'/(\d+)\.1/', href)
        if m:
            pg = int(m.group(1))
            if pg > last_page:
                last_page = pg
    return last_page


# ============================================================
# Step 10. 다음 페이지 이동 후 정렬 재적용
# ============================================================
def go_to_page_with_sort(driver, base_url, page_no, sort_value):
    """
    페이지 URL로 이동 후 정렬 select 재적용
    (라쿠텐은 페이지 이동 시 정렬이 리셋될 수 있음)
    """
    url = make_page_url(base_url, page_no)
    driver.get(url)
    time.sleep(3)
    apply_sort(driver, sort_value)
    time.sleep(2)
    return BeautifulSoup(driver.page_source, 'html.parser')


# ============================================================
# Step 11. 고평가순 수집 (앞에서부터)
# ============================================================
def collect_high(driver, base_url, target):
    collected = []
    page_no   = 1

    # 1페이지 접속 + 정렬 적용
    url = make_page_url(base_url, page_no)
    driver.get(url)
    time.sleep(3)
    apply_sort(driver, SORT_VALUE)
    time.sleep(2)

    while len(collected) < target:
        soup    = BeautifulSoup(driver.page_source, 'html.parser')
        reviews = parse_reviews(soup)

        if not reviews:
            print(f"    [고평가순] 페이지 {page_no}: 리뷰 없음 → 종료")
            break

        for r in reviews:
            if len(collected) >= target:
                break
            collected.append(r)

        print(f"    [고평가순] 페이지 {page_no}: {len(reviews)}개 → 누적 {len(collected)}/{target}")

        if len(collected) >= target or len(reviews) < 30:
            break

        # 다음 페이지 이동
        page_no += 1
        try:
            # "次のページ" 버튼 클릭 시도
            next_btn = driver.find_element(
                By.CSS_SELECTOR,
                'a[aria-label="次のページ"], a[aria-label="次へ"], a.nextPage'
            )
            driver.execute_script("arguments[0].click();", next_btn)
            time.sleep(3)
            # 정렬 재적용
            apply_sort(driver, SORT_VALUE)
            time.sleep(2)
        except:
            # 버튼 없으면 URL로 직접 이동
            soup = go_to_page_with_sort(driver, base_url, page_no, SORT_VALUE)
            continue

    return collected


# ============================================================
# Step 12. 저평가순 수집 (마지막 페이지부터 역방향)
# ============================================================
def collect_low(driver, base_url, target):
    collected = []

    # 1페이지 접속해서 마지막 페이지 파악
    url = make_page_url(base_url, 1)
    driver.get(url)
    time.sleep(3)
    apply_sort(driver, SORT_VALUE)
    time.sleep(2)
    soup      = BeautifulSoup(driver.page_source, 'html.parser')
    last_page = get_last_page(soup)
    print(f"    [저평가순] 마지막 페이지: {last_page}")

    page_no = last_page

    while len(collected) < target and page_no >= 1:
        soup = go_to_page_with_sort(driver, base_url, page_no, SORT_VALUE)
        reviews = parse_reviews(soup)

        if not reviews:
            print(f"    [저평가순] 페이지 {page_no}: 리뷰 없음 → 건너뜀")
            page_no -= 1
            continue

        # 페이지 내 역순 (가장 낮은 것부터)
        for r in reversed(reviews):
            if len(collected) >= target:
                break
            collected.append(r)

        print(f"    [저평가순] 페이지 {page_no}: {len(reviews)}개(역순) → 누적 {len(collected)}/{target}")

        if len(collected) >= target:
            break

        page_no -= 1
        time.sleep(1)

    return collected


# ============================================================
# Step 13. 전체 수집 루프 + 시트 생성
# ============================================================
wb = Workbook()
wb.remove(wb.active)

HEADER_FILL = PatternFill(fill_type='solid', fgColor='4472C4')
HEADER_FONT = Font(bold=True, color='FFFFFF', size=11)
TITLE_FILL  = PatternFill(fill_type='solid', fgColor='D9E1F2')
TITLE_FONT  = Font(bold=True, size=12)
LOW_FILL    = PatternFill(fill_type='solid', fgColor='FFF2CC')
C_ALIGN     = Alignment(horizontal='center', vertical='center', wrap_text=True)
L_ALIGN     = Alignment(horizontal='left',   vertical='top',    wrap_text=True)

COLS       = ['리뷰아이디', '별점', '리뷰내용', '정렬방식']
COL_WIDTHS = [12, 8, 90, 12]

for prod_idx, base_url in enumerate(URL_LIST, start=1):
    print(f"\n{'='*65}")
    print(f"[{prod_idx}/{len(URL_LIST)}] 상품 처리 중")

    # 상품명
    driver.get(base_url)
    time.sleep(3)
    soup = BeautifulSoup(driver.page_source, 'html.parser')
    product_name = get_product_name(soup)
    print(f"  상품명: {product_name}")

    # 고평가순 수집
    print(f"\n  ▶ 고평가순 수집 ({TARGET_PER_SORT}개 목표)")
    high_reviews = collect_high(driver, base_url, TARGET_PER_SORT)
    print(f"  ✅ 고평가순: {len(high_reviews)}개 완료")

    # 저평가순 수집
    print(f"\n  ▶ 저평가순 수집 ({TARGET_PER_SORT}개 목표)")
    low_reviews = collect_low(driver, base_url, TARGET_PER_SORT)
    print(f"  ✅ 저평가순: {len(low_reviews)}개 완료")

    # 시트 생성
    safe_name  = re.sub(r'[\\/*?:\[\]]', '', product_name)[:25]
    sheet_name = f'{prod_idx}_{safe_name}'
    ws = wb.create_sheet(title=sheet_name)

    # 행1: 상품명
    ws.merge_cells(f'A1:{get_column_letter(len(COLS))}1')
    c = ws['A1']
    c.value     = product_name
    c.fill      = TITLE_FILL
    c.font      = TITLE_FONT
    c.alignment = C_ALIGN
    ws.row_dimensions[1].height = 28

    # 행2: 헤더
    for ci, col_name in enumerate(COLS, start=1):
        c = ws.cell(row=2, column=ci, value=col_name)
        c.fill      = HEADER_FILL
        c.font      = HEADER_FONT
        c.alignment = C_ALIGN
    ws.row_dimensions[2].height = 22

    for ci, width in enumerate(COL_WIDTHS, start=1):
        ws.column_dimensions[get_column_letter(ci)].width = width

    # 데이터
    current_row = 3
    review_id   = 1

    for r in high_reviews:
        vals = [review_id, r['star'], r['review'], '고평가순']
        for ci, val in enumerate(vals, start=1):
            c = ws.cell(row=current_row, column=ci, value=val)
            c.alignment = L_ALIGN if ci == 3 else C_ALIGN
        ws.row_dimensions[current_row].height = 60
        current_row += 1
        review_id   += 1

    for r in low_reviews:
        vals = [review_id, r['star'], r['review'], '저평가순']
        for ci, val in enumerate(vals, start=1):
            c = ws.cell(row=current_row, column=ci, value=val)
            c.alignment = L_ALIGN if ci == 3 else C_ALIGN
            c.fill = LOW_FILL
        ws.row_dimensions[current_row].height = 60
        current_row += 1
        review_id   += 1

    ws.freeze_panes = 'A3'
    print(f"  시트 '{sheet_name}' 완료 (총 {review_id - 1}개)")

wb.save(fx_name)

# ============================================================
# Step 14. 요약
# ============================================================
e_time = time.time()
t_time = e_time - s_time

print("\n" + "=" * 80)
print(f"(1) 상품 수: {len(URL_LIST)}개  →  시트 {len(wb.worksheets)}개 생성")
print(f"(2) 총 소요시간: {round(t_time, 1)}초")
print(f"(3) XLSX 저장: {fx_name}")
print("=" * 80)

driver.quit()

라쿠텐 리뷰 크롤러 v5
상품 수: 11개  |  고평가순 10개 + 저평가순 10개
저장 경로: C:\py_temp\\2026-04-28-15-23-56-rakuten_review\


[1/11] 상품 처리 중
  상품명: 【楽天市場】ニキビ 吹き出物 肌荒れ 【楽天ランキング1位】 ティートリークリーム ティーツリークリーム オーガニック 保湿クリーム 大人ニキビ 思春期ニキビ 

  ▶ 고평가순 수집 (10개 목표)
    ✅ 정렬 적용 완료 (value=1)
    [고평가순] 페이지 1: 30개 → 누적 10/10
  ✅ 고평가순: 10개 완료

  ▶ 저평가순 수집 (10개 목표)
    ✅ 정렬 적용 완료 (value=1)
    [저평가순] 마지막 페이지: 1
    ✅ 정렬 적용 완료 (value=1)
    [저평가순] 페이지 1: 30개(역순) → 누적 10/10
  ✅ 저평가순: 10개 완료
  시트 '1_【楽天市場】ニキビ 吹き出物 肌荒れ 【楽天ランキ' 완료 (총 20개)

[2/11] 상품 처리 중
  상품명: 【楽天市場】idio イディオ 北海道PDRNクリーム 母の日 ギフト CICA セラミド 高保湿 乾燥 敏感肌 保湿クリーム ベタつかない 55g 日本製(F

  ▶ 고평가순 수집 (10개 목표)
    ✅ 정렬 적용 완료 (value=1)
    [고평가순] 페이지 1: 30개 → 누적 10/10
  ✅ 고평가순: 10개 완료

  ▶ 저평가순 수집 (10개 목표)
    ✅ 정렬 적용 완료 (value=1)
    [저평가순] 마지막 페이지: 1
    ✅ 정렬 적용 완료 (value=1)
    [저평가순] 페이지 1: 30개(역순) → 누적 10/10
  ✅ 저평가순: 10개 완료
  시트 '2_【楽天市場】idio イディオ 北海道PDRNクリ' 완료 (총 20개)

[3/11] 상품 처리 중
  상품명: 【楽天市場】ドレススノー ナイトクリーム 医薬部外品 レフィル＋容器＋スパチュラ/レフィル 【アテニア 公式】 [ 化粧品 美白 スキンケア エイジングケア ナ

 

In [ ]:
# ============================================================
# Step 1. 필요한 모듈 로딩
# ============================================================
from bs4 import BeautifulSoup
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import os
import re
import undetected_chromedriver as uc
from openpyxl import Workbook
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter


# ============================================================
# Step 2. 설정값
# ============================================================
URL_LIST = [
    'https://review.rakuten.co.jp/item/1/204591_10000422/1.1/?l2-id=item_review',
    'https://review.rakuten.co.jp/item/1/390266_10000002/1.1/?l2-id=item_review',
    'https://review.rakuten.co.jp/item/1/383198_10000451/1.1/?l2-id=item_review',
    'https://review.rakuten.co.jp/item/1/211527_10000085/1.1/?l2-id=item_review',
    'https://review.rakuten.co.jp/item/1/426554_10000007/1.1/?l2-id=item_review',
    'https://review.rakuten.co.jp/item/1/383198_10000231/1.1/?l2-id=item_review',
    'https://review.rakuten.co.jp/item/1/348380_10000046/1.1/?l2-id=item_review',
    'https://review.rakuten.co.jp/item/1/334402_10000099/1.1/?l2-id=item_review',
    'https://review.rakuten.co.jp/item/1/334402_10000093/1.1/?l2-id=item_review',
    'https://review.rakuten.co.jp/item/1/395258_10000116/1.1/?l2-id=item_review',
    'https://review.rakuten.co.jp/item/1/409735_10000007/1.1/?l2-id=item_review',
]

TARGET_PER_SORT = 100   # ← 본수집 시 100으로 변경
f_dir = r"C:\py_temp\\"

# select value: '1' = レビュー評価順(고평가순)
SORT_VALUE = '1'


# ============================================================
# Step 3. 저장 폴더 & 파일명 설정
# ============================================================
n = time.localtime()
s = '%04d-%02d-%02d-%02d-%02d-%02d' % (
    n.tm_year, n.tm_mon, n.tm_mday, n.tm_hour, n.tm_min, n.tm_sec
)
platform = 'rakuten_review'
out_dir  = f_dir + s + '-' + platform + '\\'
os.makedirs(out_dir, exist_ok=True)
fx_name = out_dir + s + '-' + platform + '.xlsx'

print("=" * 80)
print("라쿠텐 리뷰 크롤러 v5")
print("=" * 80)
print(f"상품 수: {len(URL_LIST)}개  |  고평가순 {TARGET_PER_SORT}개 + 저평가순 {TARGET_PER_SORT}개")
print(f"저장 경로: {out_dir}\n")


# ============================================================
# Step 4. 브라우저 실행
# ============================================================
driver = uc.Chrome(version_main=147)
driver.maximize_window()
time.sleep(2)
s_time = time.time()
wait = WebDriverWait(driver, 15)


# ============================================================
# Step 5. 핵심: 정렬 select 클릭 함수
# ============================================================
def apply_sort(driver, sort_value):
    """
    1) select 요소가 로드될 때까지 대기
    2) select를 클릭해서 드롭다운 열기
    3) 원하는 option 클릭
    4) 페이지 리로드 대기
    """
    try:
        # select 요소 대기
        select_el = wait.until(
            EC.presence_of_element_located(
                (By.CSS_SELECTOR, 'select.link-dropdown--3uu_U')
            )
        )
        # 화면에 보이도록 스크롤
        driver.execute_script("arguments[0].scrollIntoView({block:'center'});", select_el)
        time.sleep(0.5)

        # ① select 클릭 (드롭다운 열기)
        driver.execute_script("arguments[0].click();", select_el)
        time.sleep(0.5)

        # ② 원하는 option 클릭
        option = select_el.find_element(
            By.CSS_SELECTOR, f'option[value="{sort_value}"]'
        )
        driver.execute_script("arguments[0].click();", option)
        time.sleep(0.3)

        # ③ change 이벤트 강제 발생 (React 기반 UI 대응)
        driver.execute_script(
            f"""
            var sel = arguments[0];
            sel.value = '{sort_value}';
            sel.dispatchEvent(new Event('change', {{ bubbles: true }}));
            """,
            select_el
        )

        # ④ 페이지 리렌더링 대기
        time.sleep(3)
        print(f"    ✅ 정렬 적용 완료 (value={sort_value})")
        return True

    except Exception as e:
        print(f"    ⚠ 정렬 select 실패: {e}")
        return False


#
def apply_star_filter(driver, star_value):
    try:
        time.sleep(2)

        rows = driver.find_elements(By.CSS_SELECTOR, 'div.container--1G9He')

        for row in rows:
            try:
                star_text = row.find_element(
                    By.CSS_SELECTOR,
                    'div.text-display--3jedW'
                ).text.strip()

                if star_text == str(star_value):
                    btn = row.find_element(By.CSS_SELECTOR, 'button')
                    driver.execute_script("arguments[0].click();", btn)
                    time.sleep(3)
                    print(f"    ⭐ {star_value}점 필터 적용")
                    return True
            except:
                continue

        print(f"    ⚠ {star_value}점 못 찾음")
        return False

    except Exception as e:
        print(f"    ❌ 별점 필터 에러: {e}")
        return False    


# ============================================================
# Step 6. URL 생성 함수 (정렬 적용된 페이지)
# ============================================================
def make_page_url(base_url, page_no):
    match = re.search(r'/item/1/([^/]+)/', base_url)
    if not match:
        return base_url
    item_id = match.group(1)
    # 페이지 이동은 URL로, 정렬은 페이지 내 select로
    return f'https://review.rakuten.co.jp/item/1/{item_id}/{page_no}.1/'


# ============================================================
# Step 7. 리뷰 파싱 함수
# ============================================================
def parse_reviews(soup):
    reviews = []
    cards = soup.select('li > div.full-width--2Jy7N')

    for card in cards:
        # 별점
        star = '없음'
        star_el = card.select_one('span.number-wrapper-l--REPRg span')
        if star_el:
            star = star_el.get_text(strip=True)

        # 제목
        title = ''
        for tc in card.select('div[class*="type-header"]'):
            t = tc.get_text(strip=True)
            if t:
                title = t
                break

        # 본문
        body = ''
        body_el = card.select_one('div.no-ellipsis--2jV9-, div.review-body--LpVR4')
        if body_el:
            body = body_el.get_text(strip=True)

        if title and body:
            review_text = f'{title} {body}'
        elif title:
            review_text = title
        elif body:
            review_text = body
        else:
            continue

        reviews.append({'star': star, 'review': review_text.strip()})

    return reviews


# ============================================================
# Step 8. 상품명 추출
# ============================================================
def get_product_name(soup):
    try:
        crumb = soup.select('li.c-breadcrumb-list-item a')
        if len(crumb) >= 2:
            return crumb[1].get_text(strip=True)[:80]
    except:
        pass
    try:
        title = soup.select_one('title')
        if title:
            return title.get_text(strip=True).split('|')[0].strip()[:80]
    except:
        pass
    return '상품명 없음'


# ============================================================
# Step 9. 마지막 페이지 번호 파악
# ============================================================
def get_last_page(soup):
    last_page = 1
    # href 안의 페이지 번호 패턴으로 최댓값 추출
    for a in soup.select('a[href]'):
        href = a.get('href', '')
        m = re.search(r'/(\d+)\.1/', href)
        if m:
            pg = int(m.group(1))
            if pg > last_page:
                last_page = pg
    return last_page


# ============================================================
# Step 10. 다음 페이지 이동 후 정렬 재적용
# ============================================================
def go_to_page_with_sort(driver, base_url, page_no, sort_value):
    """
    페이지 URL로 이동 후 정렬 select 재적용
    (라쿠텐은 페이지 이동 시 정렬이 리셋될 수 있음)
    """
    url = make_page_url(base_url, page_no)
    driver.get(url)
    time.sleep(3)
    apply_sort(driver, sort_value)
    time.sleep(2)
    return BeautifulSoup(driver.page_source, 'html.parser')

# 다음페이지
def go_next_page(driver):
    try:
        btn = driver.find_element(By.CSS_SELECTOR, 'button[aria-label="next"]')

        # 마지막 페이지 체크
        if "navigation-button-disabled" in btn.get_attribute("class"):
            return False

        # 화면 중앙으로 이동 (중요)
        driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
        time.sleep(0.3)

        # 진짜 클릭 (이게 핵심)
        driver.execute_script("""
            arguments[0].dispatchEvent(new MouseEvent('click', {
                bubbles: true,
                cancelable: true,
                view: window
            }));
        """, btn)

        time.sleep(3)
        return True

    except:
        return False


# ============================================================
# Step 11. 고평가순 수집 (앞에서부터)
# ============================================================
def collect_high(driver, base_url, target):
    collected = []
    page_no   = 1

    # 1페이지 접속 + 정렬 적용
    url = make_page_url(base_url, page_no)
    driver.get(url)
    time.sleep(3)
    apply_sort(driver, SORT_VALUE)
    time.sleep(2)

    while len(collected) < target:
        soup    = BeautifulSoup(driver.page_source, 'html.parser')
        reviews = parse_reviews(soup)

        if not reviews:
            print(f"    [고평가순] 페이지 {page_no}: 리뷰 없음 → 종료")
            break

        for r in reviews:
            if len(collected) >= target:
                break
            collected.append(r)

        print(f"    [고평가순] 페이지 {page_no}: {len(reviews)}개 → 누적 {len(collected)}/{target}")

        if len(collected) >= target :
            break

        # 다음 페이지 이동
        try:
            next_btn = driver.find_element(
                By.CSS_SELECTOR,
                'button[aria-label="next"]'
            )

            # disabled 체크 (중요)
            if "navigation-button-disabled" in next_btn.get_attribute("class"):
                print("    ⛔ 마지막 페이지")
                break

            driver.execute_script("arguments[0].scrollIntoView({block:'center'});", next_btn)
            time.sleep(0.5)

            driver.execute_script("arguments[0].click();", next_btn)
            time.sleep(3)

        except Exception as e:
            print(f"    ❌ next 실패: {e}")
            break

    return collected


# ============================================================
# Step 12. 저평가순 수집 (마지막 페이지부터 역방향)
# ============================================================
def collect_low(driver, base_url, target):
    collected = []

    # 1점 → 2점 → 3점 순서
    for star in ['1', '2', '3','4','5']:
        print(f"\n    ▶ {star}점 리뷰 수집 시작")

        driver.get(base_url)
        time.sleep(3)

        # 별점 필터 적용
        if not apply_star_filter(driver, star):
            continue

        page_no = 1

        while len(collected) < target:
            soup = BeautifulSoup(driver.page_source, 'html.parser')
            reviews = parse_reviews(soup)

            if not reviews:
                print(f"      {star}점 p{page_no}: 리뷰 없음")
                break

            for r in reviews:
                if len(collected) >= target:
                    break
                collected.append(r)

            print(f"      {star}점 p{page_no}: {len(reviews)}개 → 누적 {len(collected)}/{target}")

            # 다음 페이지 이동 (필터 유지됨)
            try:
                next_btns = driver.find_elements(By.CSS_SELECTOR, 'button[aria-label="next"]')

                if not next_btns:
                    print(f"      ⛔ 마지막 페이지 도달")
                    break

                btn = next_btns[0]

                if "navigation-button-disabled" in btn.get_attribute("class"):
                    print(f"      ⛔ 마지막 페이지 도달")
                    break

                old_html = driver.page_source

                driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
                time.sleep(0.3)

                driver.execute_script("""
                    arguments[0].dispatchEvent(new MouseEvent('click', {
                        bubbles: true,
                        cancelable: true,
                        view: window
                    }));
                """, btn)

                time.sleep(3)

                new_html = driver.page_source
                if old_html == new_html:
                    print("      ⚠ 페이지 안 변함 → 종료")
                    break

                page_no += 1

            except Exception as e:
                print(f"      ❌ next 실패: {e}")
                break

        if len(collected) >= target:
            break

    return collected


# ============================================================
# Step 13. 전체 수집 루프 + 시트 생성
# ============================================================
wb = Workbook()
wb.remove(wb.active)

HEADER_FILL = PatternFill(fill_type='solid', fgColor='4472C4')
HEADER_FONT = Font(bold=True, color='FFFFFF', size=11)
TITLE_FILL  = PatternFill(fill_type='solid', fgColor='D9E1F2')
TITLE_FONT  = Font(bold=True, size=12)
LOW_FILL    = PatternFill(fill_type='solid', fgColor='FFF2CC')
C_ALIGN     = Alignment(horizontal='center', vertical='center', wrap_text=True)
L_ALIGN     = Alignment(horizontal='left',   vertical='top',    wrap_text=True)

COLS       = ['리뷰아이디', '별점', '리뷰내용', '정렬방식']
COL_WIDTHS = [12, 8, 90, 12]

for prod_idx, base_url in enumerate(URL_LIST, start=1):
    print(f"\n{'='*65}")
    print(f"[{prod_idx}/{len(URL_LIST)}] 상품 처리 중")

    # 상품명
    driver.get(base_url)
    time.sleep(3)
    soup = BeautifulSoup(driver.page_source, 'html.parser')
    product_name = get_product_name(soup)
    print(f"  상품명: {product_name}")

    # 고평가순 수집
    print(f"\n  ▶ 고평가순 수집 ({TARGET_PER_SORT}개 목표)")
    high_reviews = collect_high(driver, base_url, TARGET_PER_SORT)
    print(f"  ✅ 고평가순: {len(high_reviews)}개 완료")

    # 저평가순 수집
    print(f"\n  ▶ 저평가순 수집 ({TARGET_PER_SORT}개 목표)")
    low_reviews = collect_low(driver, base_url, TARGET_PER_SORT)
    print(f"  ✅ 저평가순: {len(low_reviews)}개 완료")

    # 시트 생성
    safe_name  = re.sub(r'[\\/*?:\[\]]', '', product_name)[:25]
    sheet_name = f'{prod_idx}_{safe_name}'
    ws = wb.create_sheet(title=sheet_name)

    # 행1: 상품명
    ws.merge_cells(f'A1:{get_column_letter(len(COLS))}1')
    c = ws['A1']
    c.value     = product_name
    c.fill      = TITLE_FILL
    c.font      = TITLE_FONT
    c.alignment = C_ALIGN
    ws.row_dimensions[1].height = 28

    # 행2: 헤더
    for ci, col_name in enumerate(COLS, start=1):
        c = ws.cell(row=2, column=ci, value=col_name)
        c.fill      = HEADER_FILL
        c.font      = HEADER_FONT
        c.alignment = C_ALIGN
    ws.row_dimensions[2].height = 22

    for ci, width in enumerate(COL_WIDTHS, start=1):
        ws.column_dimensions[get_column_letter(ci)].width = width

    # 데이터
    current_row = 3
    review_id   = 1

    for idx, r in enumerate(high_reviews, start=1):
        vals = [review_id, r['star'], r['review'], f'고평가순 {idx}번째']
        for ci, val in enumerate(vals, start=1):
            c = ws.cell(row=current_row, column=ci, value=val)
            c.alignment = L_ALIGN if ci == 3 else C_ALIGN
        ws.row_dimensions[current_row].height = 60
        current_row += 1
        review_id   += 1

    for idx, r in enumerate(low_reviews, start=1):
        vals = [review_id, r['star'], r['review'], f'저평가순 {idx}번째']
        for ci, val in enumerate(vals, start=1):
            c = ws.cell(row=current_row, column=ci, value=val)
            c.alignment = L_ALIGN if ci == 3 else C_ALIGN
            c.fill = LOW_FILL
        ws.row_dimensions[current_row].height = 60
        current_row += 1
        review_id   += 1

    ws.freeze_panes = 'A3'
    print(f"  시트 '{sheet_name}' 완료 (총 {review_id - 1}개)")

wb.save(fx_name)

# ============================================================
# Step 14. 요약
# ============================================================
e_time = time.time()
t_time = e_time - s_time

print("\n" + "=" * 80)
print(f"(1) 상품 수: {len(URL_LIST)}개  →  시트 {len(wb.worksheets)}개 생성")
print(f"(2) 총 소요시간: {round(t_time, 1)}초")
print(f"(3) XLSX 저장: {fx_name}")
print("=" * 80)


라쿠텐 리뷰 크롤러 v5
상품 수: 11개  |  고평가순 100개 + 저평가순 100개
저장 경로: C:\py_temp\\2026-04-28-17-07-58-rakuten_review\


[1/11] 상품 처리 중
  상품명: 【楽天市場】ニキビ 吹き出物 肌荒れ 【楽天ランキング1位】 ティートリークリーム ティーツリークリーム オーガニック 保湿クリーム 大人ニキビ 思春期ニキビ 

  ▶ 고평가순 수집 (100개 목표)
    ✅ 정렬 적용 완료 (value=1)
    [고평가순] 페이지 1: 30개 → 누적 30/100
    [고평가순] 페이지 1: 30개 → 누적 60/100
    [고평가순] 페이지 1: 30개 → 누적 90/100
    [고평가순] 페이지 1: 30개 → 누적 100/100
  ✅ 고평가순: 100개 완료

  ▶ 저평가순 수집 (100개 목표)

    ▶ 1점 리뷰 수집 시작
    ⭐ 1점 필터 적용
      1점 p1: 30개 → 누적 30/100
      1점 p2: 30개 → 누적 60/100
      1점 p3: 11개 → 누적 71/100
      ⛔ 마지막 페이지 도달

    ▶ 2점 리뷰 수집 시작
    ⭐ 2점 필터 적용
      2점 p1: 30개 → 누적 100/100
  ✅ 저평가순: 100개 완료
  시트 '1_【楽天市場】ニキビ 吹き出物 肌荒れ 【楽天ランキ' 완료 (총 200개)

[2/11] 상품 처리 중
  상품명: 【楽天市場】idio イディオ 北海道PDRNクリーム 母の日 ギフト CICA セラミド 高保湿 乾燥 敏感肌 保湿クリーム ベタつかない 55g 日本製(F

  ▶ 고평가순 수집 (100개 목표)
    ✅ 정렬 적용 완료 (value=1)
    [고평가순] 페이지 1: 30개 → 누적 30/100
    [고평가순] 페이지 1: 30개 → 누적 60/100
    [고평가순] 페이지 1: 30개 → 누적 90/100
    [고평가순] 페이지 1: 30개 → 누적 100/100
  ✅ 고평

# 찐최종

In [17]:
# ============================================================
# Step 1. 필요한 모듈 로딩
# ============================================================
from bs4 import BeautifulSoup
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import os
import re
import undetected_chromedriver as uc
from openpyxl import Workbook
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter
 
 
# ============================================================
# Step 2. 설정값
# ============================================================
URL_LIST = [
    'https://review.rakuten.co.jp/item/1/204591_10000422/1.1/?l2-id=item_review',
    'https://review.rakuten.co.jp/item/1/390266_10000002/1.1/?l2-id=item_review',
    'https://review.rakuten.co.jp/item/1/383198_10000451/1.1/?l2-id=item_review',
    'https://review.rakuten.co.jp/item/1/211527_10000085/1.1/?l2-id=item_review',
    'https://review.rakuten.co.jp/item/1/426554_10000007/1.1/?l2-id=item_review',
    'https://review.rakuten.co.jp/item/1/383198_10000231/1.1/?l2-id=item_review',
    'https://review.rakuten.co.jp/item/1/348380_10000046/1.1/?l2-id=item_review',
    'https://review.rakuten.co.jp/item/1/334402_10000099/1.1/?l2-id=item_review',
    'https://review.rakuten.co.jp/item/1/334402_10000093/1.1/?l2-id=item_review',
    'https://review.rakuten.co.jp/item/1/395258_10000116/1.1/?l2-id=item_review',
    'https://review.rakuten.co.jp/item/1/409735_10000007/1.1/?l2-id=item_review',
]
 
TARGET_PER_SORT = 100
f_dir = r"C:\py_temp\\"
 
# select value: '1' = レビュー評価順(고평가순)
SORT_VALUE = '1'
 
 
# ============================================================
# Step 3. 저장 폴더 & 파일명 설정
# ============================================================
n = time.localtime()
s = '%04d-%02d-%02d-%02d-%02d-%02d' % (
    n.tm_year, n.tm_mon, n.tm_mday, n.tm_hour, n.tm_min, n.tm_sec
)
platform = 'rakuten_review'
out_dir  = f_dir + s + '-' + platform + '\\'
os.makedirs(out_dir, exist_ok=True)
fx_name = out_dir + s + '-' + platform + '.xlsx'
 
print("=" * 80)
print("라쿠텐 리뷰 크롤러 v6")
print("=" * 80)
print(f"상품 수: {len(URL_LIST)}개  |  고평가순 {TARGET_PER_SORT}개 + 저평가순 {TARGET_PER_SORT}개")
print(f"저장 경로: {out_dir}\n")
 
 
# ============================================================
# Step 4. 브라우저 실행
# ============================================================
driver = uc.Chrome(version_main=147)
driver.maximize_window()
time.sleep(2)
s_time = time.time()
wait = WebDriverWait(driver, 15)
 
 
# ============================================================
# Step 5. 정렬 select 클릭 함수
# ============================================================
def apply_sort(driver, sort_value):
    try:
        select_el = wait.until(
            EC.presence_of_element_located(
                (By.CSS_SELECTOR, 'select.link-dropdown--3uu_U')
            )
        )
        driver.execute_script("arguments[0].scrollIntoView({block:'center'});", select_el)
        time.sleep(0.5)
 
        driver.execute_script("arguments[0].click();", select_el)
        time.sleep(0.5)
 
        option = select_el.find_element(
            By.CSS_SELECTOR, f'option[value="{sort_value}"]'
        )
        driver.execute_script("arguments[0].click();", option)
        time.sleep(0.3)
 
        driver.execute_script(
            f"""
            var sel = arguments[0];
            sel.value = '{sort_value}';
            sel.dispatchEvent(new Event('change', {{ bubbles: true }}));
            """,
            select_el
        )
 
        time.sleep(3)
        print(f"    ✅ 정렬 적용 완료 (value={sort_value})")
        return True
 
    except Exception as e:
        print(f"    ⚠ 정렬 select 실패: {e}")
        return False
 
 
# ============================================================
# Step 6. 별점 필터 함수
# ============================================================
def apply_star_filter(driver, star_value):
    try:
        time.sleep(2)

        rows = driver.find_elements(By.CSS_SELECTOR, 'div.container--1G9He')

        for row in rows:
            try:
                star_text = row.find_element(
                    By.CSS_SELECTOR,
                    'div.text-display--3jedW'
                ).text.strip()

                if star_text == str(star_value):
                    btn = row.find_element(By.CSS_SELECTOR, 'button')

                    # 0건이면 disabled → 스킵
                    if btn.get_attribute('disabled'):
                        print(f"    ⚠ {star_value}점 리뷰 0건 → 스킵")
                        return False

                    driver.execute_script("arguments[0].click();", btn)
                    time.sleep(3)
                    print(f"    ⭐ {star_value}점 필터 적용")
                    return True
            except:
                continue

        print(f"    ⚠ {star_value}점 못 찾음")
        return False

    except Exception as e:
        print(f"    ❌ 별점 필터 에러: {e}")
        return False
    
# ============================================================
# Step 7. URL 생성 함수
# ============================================================
def make_page_url(base_url, page_no):
    match = re.search(r'/item/1/([^/]+)/', base_url)
    if not match:
        return base_url
    item_id = match.group(1)
    return f'https://review.rakuten.co.jp/item/1/{item_id}/{page_no}.1/'
 
 
# ============================================================
# Step 8. 리뷰 파싱 함수
# ============================================================
def parse_reviews(soup):
    reviews = []
    cards = soup.select('li > div.full-width--2Jy7N')
 
    for card in cards:
        # 별점
        star = '없음'
        star_el = card.select_one('span.number-wrapper-l--REPRg span')
        if star_el:
            star = star_el.get_text(strip=True)
 
        # 제목
        title = ''
        for tc in card.select('div[class*="type-header"]'):
            t = tc.get_text(strip=True)
            if t:
                title = t
                break
 
        # 본문
        body = ''
        body_el = card.select_one('div.no-ellipsis--2jV9-, div.review-body--LpVR4')
        if body_el:
            body = body_el.get_text(strip=True)
 
        if title and body:
            review_text = f'{title} {body}'
        elif title:
            review_text = title
        elif body:
            review_text = body
        else:
            continue
 
        reviews.append({'star': star, 'review': review_text.strip()})
 
    return reviews
 
 
# ============================================================
# Step 9. 상품명 추출
# ============================================================
def get_product_name(soup):
    try:
        crumb = soup.select('li.c-breadcrumb-list-item a')
        if len(crumb) >= 2:
            return crumb[1].get_text(strip=True)[:80]
    except:
        pass
    try:
        title = soup.select_one('title')
        if title:
            return title.get_text(strip=True).split('|')[0].strip()[:80]
    except:
        pass
    return '상품명 없음'
 
 
# ============================================================
# Step 10. 고평가순 수집
# ============================================================
def collect_high(driver, base_url, target, seen_set):
    collected = []
    page_no = 1
 
    url = make_page_url(base_url, page_no)
    driver.get(url)
    time.sleep(3)
    apply_sort(driver, SORT_VALUE)
    time.sleep(2)
 
    while len(collected) < target:
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        reviews = parse_reviews(soup)
 
        if not reviews:
            print(f"    [고평가순] 페이지 {page_no}: 리뷰 없음 → 종료")
            break
 
        for r in reviews:
            if len(collected) >= target:
                break
            if r['review'] in seen_set:  # 중복 스킵 → 다른 리뷰 수집
                continue
            seen_set.add(r['review'])
            collected.append(r)
 
        print(f"    [고평가순] 페이지 {page_no}: {len(reviews)}개 파싱 → 누적 {len(collected)}/{target}")
 
        if len(collected) >= target:
            break
 
        # 다음 페이지 이동
        try:
            next_btn = driver.find_element(
                By.CSS_SELECTOR,
                'button[aria-label="next"]'
            )
 
            if "navigation-button-disabled" in next_btn.get_attribute("class"):
                print("    ⛔ 마지막 페이지")
                break
 
            driver.execute_script("arguments[0].scrollIntoView({block:'center'});", next_btn)
            time.sleep(0.5)
            driver.execute_script("arguments[0].click();", next_btn)
            time.sleep(3)
            page_no += 1
 
        except Exception as e:
            print(f"    ❌ next 실패: {e}")
            break
 
    return collected
 
 
# ============================================================
# Step 11. 저평가순 수집 (1점→2점→3점→4점→5점, 100개 채울 때까지)
# ============================================================
def collect_low(driver, base_url, target, seen_set):
    collected = []
 
    # 1점부터 5점까지 순서대로, target 채울 때까지 계속
    for star in ['1', '2', '3', '4', '5']:
        if len(collected) >= target:
            break
 
        print(f"\n    ▶ {star}점 리뷰 수집 시작")
 
        driver.get(base_url)
        time.sleep(3)
 
        if not apply_star_filter(driver, star):
            continue
 
        page_no = 1
 
        while len(collected) < target:
            soup = BeautifulSoup(driver.page_source, 'html.parser')
            reviews = parse_reviews(soup)
 
            if not reviews:
                print(f"      {star}점 p{page_no}: 리뷰 없음")
                break
 
            added_this_page = 0
            for r in reviews:
                if len(collected) >= target:
                    break
                if r['review'] in seen_set:  # 중복 스킵 → 다른 리뷰 수집
                    continue
                seen_set.add(r['review'])
                collected.append(r)
                added_this_page += 1
 
            print(f"      {star}점 p{page_no}: {len(reviews)}개 파싱 / {added_this_page}개 신규 → 누적 {len(collected)}/{target}")
 
            # 다음 페이지 이동
            try:
                next_btns = driver.find_elements(By.CSS_SELECTOR, 'button[aria-label="next"]')
 
                if not next_btns:
                    print(f"      ⛔ 마지막 페이지 도달")
                    break
 
                btn = next_btns[0]
 
                if "navigation-button-disabled" in btn.get_attribute("class"):
                    print(f"      ⛔ 마지막 페이지 도달")
                    break
 
                old_html = driver.page_source
 
                driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
                time.sleep(0.3)
 
                driver.execute_script("""
                    arguments[0].dispatchEvent(new MouseEvent('click', {
                        bubbles: true,
                        cancelable: true,
                        view: window
                    }));
                """, btn)
 
                time.sleep(3)
 
                new_html = driver.page_source
                if old_html == new_html:
                    print("      ⚠ 페이지 안 변함 → 종료")
                    break
 
                page_no += 1
 
            except Exception as e:
                print(f"      ❌ next 실패: {e}")
                break
 
    return collected
 
 
# ============================================================
# Step 12. 전체 수집 루프 + 시트 생성
# ============================================================
wb = Workbook()
wb.remove(wb.active)
 
HEADER_FILL = PatternFill(fill_type='solid', fgColor='4472C4')
HEADER_FONT = Font(bold=True, color='FFFFFF', size=11)
TITLE_FILL  = PatternFill(fill_type='solid', fgColor='D9E1F2')
TITLE_FONT  = Font(bold=True, size=12)
LOW_FILL    = PatternFill(fill_type='solid', fgColor='FFF2CC')
C_ALIGN     = Alignment(horizontal='center', vertical='center', wrap_text=True)
L_ALIGN     = Alignment(horizontal='left',   vertical='top',    wrap_text=True)
 
COLS       = ['리뷰아이디', '별점', '리뷰내용', '정렬방식']
COL_WIDTHS = [12, 8, 90, 12]
 
for prod_idx, base_url in enumerate(URL_LIST, start=1):
    print(f"\n{'='*65}")
    print(f"[{prod_idx}/{len(URL_LIST)}] 상품 처리 중")
 
    # 상품명
    driver.get(base_url)
    time.sleep(3)
    soup = BeautifulSoup(driver.page_source, 'html.parser')
    product_name = get_product_name(soup)
    print(f"  상품명: {product_name}")
 
    # 상품별 seen_set (고평가↔저평가 공유 → 교차 중복 방지)
    seen_reviews = set()
 
    # 고평가순 수집
    print(f"\n  ▶ 고평가순 수집 ({TARGET_PER_SORT}개 목표)")
    high_reviews = collect_high(driver, base_url, TARGET_PER_SORT, seen_reviews)
    print(f"  ✅ 고평가순: {len(high_reviews)}개 완료")
 
    # 저평가순 수집 (1→2→3→4→5점, 중복 제외하고 100개 채울 때까지)
    print(f"\n  ▶ 저평가순 수집 ({TARGET_PER_SORT}개 목표)")
    low_reviews = collect_low(driver, base_url, TARGET_PER_SORT, seen_reviews)
    print(f"  ✅ 저평가순: {len(low_reviews)}개 완료")
 
    # 시트 생성
    safe_name  = re.sub(r'[\\/*?:\[\]]', '', product_name)[:25]
    sheet_name = f'{prod_idx}_{safe_name}'
    ws = wb.create_sheet(title=sheet_name)
 
    # 행1: 상품명
    ws.merge_cells(f'A1:{get_column_letter(len(COLS))}1')
    c = ws['A1']
    c.value     = product_name
    c.fill      = TITLE_FILL
    c.font      = TITLE_FONT
    c.alignment = C_ALIGN
    ws.row_dimensions[1].height = 28
 
    # 행2: 헤더
    for ci, col_name in enumerate(COLS, start=1):
        c = ws.cell(row=2, column=ci, value=col_name)
        c.fill      = HEADER_FILL
        c.font      = HEADER_FONT
        c.alignment = C_ALIGN
    ws.row_dimensions[2].height = 22
 
    for ci, width in enumerate(COL_WIDTHS, start=1):
        ws.column_dimensions[get_column_letter(ci)].width = width
 
    # 데이터 입력
    current_row = 3
    review_id   = 1
 
    for idx, r in enumerate(high_reviews, start=1):
        vals = [review_id, r['star'], r['review'], f'고평가순 {idx}번째']
        for ci, val in enumerate(vals, start=1):
            c = ws.cell(row=current_row, column=ci, value=val)
            c.alignment = L_ALIGN if ci == 3 else C_ALIGN
        ws.row_dimensions[current_row].height = 60
        current_row += 1
        review_id   += 1
 
    for idx, r in enumerate(low_reviews, start=1):
        vals = [review_id, r['star'], r['review'], f'저평가순 {idx}번째']
        for ci, val in enumerate(vals, start=1):
            c = ws.cell(row=current_row, column=ci, value=val)
            c.alignment = L_ALIGN if ci == 3 else C_ALIGN
            c.fill = LOW_FILL
        ws.row_dimensions[current_row].height = 60
        current_row += 1
        review_id   += 1
 
    ws.freeze_panes = 'A3'
    print(f"  시트 '{sheet_name}' 완료 (총 {review_id - 1}개)")
 
wb.save(fx_name)
 
# ============================================================
# Step 13. 요약
# ============================================================
e_time = time.time()
t_time = e_time - s_time
 
print("\n" + "=" * 80)
print(f"(1) 상품 수: {len(URL_LIST)}개  →  시트 {len(wb.worksheets)}개 생성")
print(f"(2) 총 소요시간: {round(t_time, 1)}초")
print(f"(3) XLSX 저장: {fx_name}")
print("=" * 80)

라쿠텐 리뷰 크롤러 v6
상품 수: 11개  |  고평가순 100개 + 저평가순 100개
저장 경로: C:\py_temp\\2026-04-28-18-05-13-rakuten_review\


[1/11] 상품 처리 중
  상품명: 【楽天市場】ニキビ 吹き出物 肌荒れ 【楽天ランキング1位】 ティートリークリーム ティーツリークリーム オーガニック 保湿クリーム 大人ニキビ 思春期ニキビ 

  ▶ 고평가순 수집 (100개 목표)
    ✅ 정렬 적용 완료 (value=1)
    [고평가순] 페이지 1: 30개 파싱 → 누적 30/100
    [고평가순] 페이지 2: 30개 파싱 → 누적 60/100
    [고평가순] 페이지 3: 30개 파싱 → 누적 90/100
    [고평가순] 페이지 4: 30개 파싱 → 누적 100/100
  ✅ 고평가순: 100개 완료

  ▶ 저평가순 수집 (100개 목표)

    ▶ 1점 리뷰 수집 시작
    ⭐ 1점 필터 적용
      1점 p1: 30개 파싱 / 30개 신규 → 누적 30/100
      1점 p2: 30개 파싱 / 30개 신규 → 누적 60/100
      1점 p3: 11개 파싱 / 11개 신규 → 누적 71/100
      ⛔ 마지막 페이지 도달

    ▶ 2점 리뷰 수집 시작
    ⭐ 2점 필터 적용
      2점 p1: 30개 파싱 / 29개 신규 → 누적 100/100
  ✅ 저평가순: 100개 완료
  시트 '1_【楽天市場】ニキビ 吹き出物 肌荒れ 【楽天ランキ' 완료 (총 200개)

[2/11] 상품 처리 중
  상품명: 【楽天市場】idio イディオ 北海道PDRNクリーム 母の日 ギフト CICA セラミド 高保湿 乾燥 敏感肌 保湿クリーム ベタつかない 55g 日本製(F

  ▶ 고평가순 수집 (100개 목표)
    ✅ 정렬 적용 완료 (value=1)
    [고평가순] 페이지 1: 30개 파싱 → 누적 29/100
    [고평가순] 페이지 2: 30개 파싱 → 누적 59/100
    [고평가순